We start by examining an available dataset that is similar to our needs

In [1]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("tomekkorbak/python-github-code", split="test")
ds.set_format("pandas")
ds

Dataset({
    features: ['text', 'repo_name', 'path', 'language', 'license', 'size', 'score'],
    num_rows: 10000
})

The dataset contains

"text":  content of python file

"size":  number of characters in the file

"score": not identified by dataset card so we will ignore it

In [2]:
df = ds[0:10]
df

,text,repo_name,path,language,license,size,score
0,from __future__ import print_function\r\n\r\ni...,Jeff-Tian/mybnb,Python27/Lib/test/test___all__.py,Python,apache-2.0,4298,0.000931
1,"# (c) 2012-2014, Michael DeHaan <michael.dehaa...",bootswithdefer/ansible,v2/ansible/playbook/__init__.py,Python,gpl-3.0,3162,0.002214
2,from __future__ import print_function\nfrom pa...,dssg/wikienergy,disaggregator/build/pandas/pandas/io/tests/tes...,Python,mit,17790,0.001518
3,"""""""\nOutlier Detection using Tukeys Filter Cla...",trademob/anna-molly,lib/plugins/tukeys_filter.py,Python,mit,5255,0.001713
4,## -*- Mode: python; py-indent-offset: 4; inde...,srene/ndnSIM-inrpp,src/ndnSIM/examples/ndn-simple.py,Python,gpl-2.0,3286,0.005782
5,# Copyright (c) 2018 PaddlePaddle Authors. A...,PaddlePaddle/Paddle,python/paddle/fluid/tests/unittests/xpu/test_r...,Python,apache-2.0,6221,0.000643
6,# -*- coding: utf-8 -*-\nfrom __future__ impor...,marevol/gym-starter-kit-example,pong/__init__.py,Python,apache-2.0,240,0.004167
7,"from tasks import func\n\nfunc.delay(1, 2)\n",emanuelvianna/microcelery,tests/client.py,Python,gpl-2.0,41,0.000000
8,import re\n\nfrom tornado.escape import xhtml_...,JmPotato/College,handlers/utils.py,Python,mit,2237,0.000447
9,"# Copyright (c) 2013, Web Notes Technologies P...",cadencewatches/frappe,frappe/core/doctype/event/test_event.py,Python,mit,1564,0.021100


We use [Pylint](https://pylint.readthedocs.io/en/stable/index.html) to get an initial score of python file quality

In [3]:
import subprocess
import tempfile

def get_pylint_text(row: pd.Series) -> str:
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".py",
        delete=False
    ) as f:
        f.write(row["text"])
        filename = f.name

    result = subprocess.run(
        ["pylint", filename, "-r", "n", "--output-format=json2"],
        capture_output=True,
        text=True
    )

    return result.stdout

df["pylint_text"] = df.apply(get_pylint_text, axis=1)

Examining Pylint output shows that files with "fatal" or "error" messages do not even run and thus get zero score

This is good since we do not want to give a score to a python file that does not even run in the first place

Examining [Pylint messages](https://pylint.readthedocs.io/en/stable/user_guide/messages/messages_overview.html#) more closely shows that all fatal and error messages are legit to consider that the file won't run except [import-error / E0401](https://pylint.readthedocs.io/en/stable/user_guide/messages/error/import-error.html) which would be hard to verify in deployment since it would require creating a venv then downloading the file's dependencies so we will simply ignore it

We do so by running

    pylint --generate-rcfile > .pylintrc

in terminal which would create the config file ".pylintrc" then adding "disabl=E0401" under "[MESSAGES CONTROL]" in that file

In [4]:
import json

sample_pylint_text = json.loads(df["pylint_text"][0])
sample_pylint_text

{'messages': [{'type': 'error',
   'symbol': 'syntax-error',
   'message': "Parsing failed: 'Missing parentheses in call to 'exec'. Did you mean exec(...)? (tmpcmtosncs, line 29)'",
   'messageId': 'E0001',
   'confidence': 'HIGH',
   'module': 'tmpcmtosncs',
   'obj': '',
   'line': 29,
   'column': 17,
   'endLine': None,
   'endColumn': None,
   'path': '/tmp/tmpcmtosncs.py',
   'absolutePath': '/tmp/tmpcmtosncs.py'}],
 'statistics': {'messageTypeCount': {'fatal': 0,
   'error': 1,
   'warning': 0,
   'refactor': 0,
   'convention': 0,
   'info': 0},
  'modulesLinted': 4,
  'score': 0}}

We extract score where files with fatal or error messages get -1 (representing Null) and others get thier Pylint score

In [5]:
import json

def get_pylint_score(row: pd.Series) -> float:
    pylint_json = json.loads(row["pylint_text"])
    fatal = pylint_json["statistics"]["messageTypeCount"]["fatal"]
    error = pylint_json["statistics"]["messageTypeCount"]["error"]

    if fatal or error:
        return -1
    else:
        return pylint_json["statistics"]["score"]

df["pylint_score"] = df.apply(get_pylint_score, axis=1)

In [6]:
df

,text,repo_name,path,language,license,size,score,pylint_text,pylint_score
0,from __future__ import print_function\r\n\r\ni...,Jeff-Tian/mybnb,Python27/Lib/test/test___all__.py,Python,apache-2.0,4298,0.000931,"{\n ""messages"": [\n {\n ""...",-1.000000
1,"# (c) 2012-2014, Michael DeHaan <michael.dehaa...",bootswithdefer/ansible,v2/ansible/playbook/__init__.py,Python,gpl-3.0,3162,0.002214,"{\n ""messages"": [\n {\n ""...",6.250000
2,from __future__ import print_function\nfrom pa...,dssg/wikienergy,disaggregator/build/pandas/pandas/io/tests/tes...,Python,mit,17790,0.001518,"{\n ""messages"": [\n {\n ""...",-1.000000
3,"""""""\nOutlier Detection using Tukeys Filter Cla...",trademob/anna-molly,lib/plugins/tukeys_filter.py,Python,mit,5255,0.001713,"{\n ""messages"": [\n {\n ""...",7.634409
4,## -*- Mode: python; py-indent-offset: 4; inde...,srene/ndnSIM-inrpp,src/ndnSIM/examples/ndn-simple.py,Python,gpl-2.0,3286,0.005782,"{\n ""messages"": [\n {\n ""...",-1.000000
5,# Copyright (c) 2018 PaddlePaddle Authors. A...,PaddlePaddle/Paddle,python/paddle/fluid/tests/unittests/xpu/test_r...,Python,apache-2.0,6221,0.000643,"{\n ""messages"": [\n {\n ""...",7.037037
6,# -*- coding: utf-8 -*-\nfrom __future__ impor...,marevol/gym-starter-kit-example,pong/__init__.py,Python,apache-2.0,240,0.004167,"{\n ""messages"": [\n {\n ""...",8.000000
7,"from tasks import func\n\nfunc.delay(1, 2)\n",emanuelvianna/microcelery,tests/client.py,Python,gpl-2.0,41,0.000000,"{\n ""messages"": [\n {\n ""...",5.000000
8,import re\n\nfrom tornado.escape import xhtml_...,JmPotato/College,handlers/utils.py,Python,mit,2237,0.000447,"{\n ""messages"": [\n {\n ""...",7.872340
9,"# Copyright (c) 2013, Web Notes Technologies P...",cadencewatches/frappe,frappe/core/doctype/event/test_event.py,Python,mit,1564,0.021100,"{\n ""messages"": [\n {\n ""...",-1.000000
